# Session 2 Lab — Build an SMS Spam Classifier

**Beginner core + optional advanced challenge**  ·  **Environment:** Google Colab, CPU runtime

**Learning goal:** Investigate data → split → train → evaluate → test custom messages → save a complete model.

**Classroom routine:** **WHY?** Ask a question → **RUN** a cell → **OBSERVE** the output → **EXPLAIN** it.

**Dataset:** [UCI SMS Spam Collection](https://archive.ics.uci.edu/dataset/228/sms+spam+collection) (5,574 labeled messages in the source file). This public research dataset is historical and does not reflect all modern SMS spam.

**Privacy:** Use *invented* SMS for the custom-message exercise, never personal messages. **Safety:** Do not load `.joblib` files from untrusted sources.

## Step 0 — Is our environment ready?
**WHY?** The notebook needs pandas for data and scikit-learn for ML.

**Ask:** Do we need to install an IDE, Git, or Docker for this lab? **No.** Colab provides Python; its standard CPU runtime is enough.

**RUN:** Check that the packages import. **EXPECT:** Two version numbers; no errors.

In [ ]:
import pandas as pd
import sklearn
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)

## Step 1 — Load the real SMS dataset
**WHY?** A classifier must learn from labeled examples.

**Ask before running:** What should one dataset row contain?

**RUN:** Download the original dataset from UCI, or upload its ZIP/extracted `SMSSpamCollection` file if the network blocks it. The notebook creates two columns: `label` and `text`.

**EXPECT:** **5,574** original records, before cleaning. Download problems? Use the [UCI dataset page](https://archive.ics.uci.edu/dataset/228/sms+spam+collection) to download the ZIP first.

In [ ]:
from io import BytesIO
from zipfile import ZipFile
from urllib.request import urlopen
import csv

DATA_URL = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"

def read_uci_zip(data):
    with ZipFile(BytesIO(data)) as z:
        match = next((name for name in z.namelist()
                      if name.rsplit("/", 1)[-1] == "SMSSpamCollection"), None)
        if match is None:
            raise ValueError("ZIP does not contain SMSSpamCollection")
        return z.read(match)

try:
    with urlopen(DATA_URL, timeout=20) as response:
        raw_sms = read_uci_zip(response.read())
    print("Downloaded original dataset from UCI")
except Exception as error:
    print("Automatic download failed:", str(error)[:130])
    print("Upload the original UCI ZIP or its extracted SMSSpamCollection file.")
    from google.colab import files
    uploaded = files.upload()
    name, content = next(iter(uploaded.items()))
    raw_sms = read_uci_zip(content) if name.lower().endswith(".zip") else content

sms = pd.read_csv(
    BytesIO(raw_sms), sep="\t", header=None,
    names=["label", "text"], encoding="latin-1", quoting=csv.QUOTE_NONE
)
print("Original records:", len(sms))
assert len(sms) == 5574, "Unexpected source size; verify the original UCI file"
assert set(sms["label"].dropna().unique()) == {"ham", "spam"}
display(sms.head(3))

## Step 2 — Guided dataset investigation
**WHY?** Good ML work starts by understanding the examples, not by immediately calling `fit()`.

Discuss these questions **before** running the next cells:
1. What does one row represent?
2. Which column is the **input**? Which is the **label**?
3. Are ham and spam equally common?
4. What accuracy would an **always-ham** classifier obtain on this raw dataset?
5. Which words or patterns might appear in spam? Could legitimate messages contain them too?

**RUN:** Examine the shape, labels, and a few examples.

In [ ]:
print("Rows, columns:", sms.shape)
print("Column names:", sms.columns.tolist())
print("Class counts:")
print(sms["label"].value_counts())
print("Class proportions (%):")
print((sms["label"].value_counts(normalize=True) * 100).round(1))

### Guided Exercise 1 — Investigate messages
**Your turn:** Change `selected_label` to `"spam"`, then run the cell again.

**Ask:** Can a single word, such as *free*, always tell us the correct label? Why not?

**Hint:** Valid label values are `"ham"` and `"spam"`. This displays examples from the public dataset; don't enter personal SMS.

In [ ]:
selected_label = "ham"  # YOUR TURN: change to "spam" and rerun
examples = sms.loc[sms["label"] == selected_label, ["label", "text"]]
display(examples.head(4))

### Guided Exercise 2 — Count each category
**Your turn:** Uncomment the second line to display both label counts.

**Think:** Why could a model achieve high accuracy by always saying `ham`?

In [ ]:
print("Total messages:", len(sms))
# print(sms["label"].value_counts())  # YOUR TURN: remove the # and rerun

## Step 3 — Prepare and split the dataset
**WHY?** The test data should be new to the model. Duplicate texts can make a split misleading.

We remove empty messages and exact repeated texts **before** splitting. Some records will disappear — that's expected.

Then we create **training**, **validation**, and **test** sets. Each keeps approximately the same ham/spam ratio.

**REMEMBER:** Training = learn; validation = choose/check; test = final exam.

In [ ]:
from sklearn.model_selection import train_test_split

original_count = len(sms)
sms = sms.dropna(subset=["label", "text"]).copy()
sms["text"] = sms["text"].astype(str).str.strip()
sms = sms[sms["text"] != ""].drop_duplicates(subset="text").reset_index(drop=True)
print("Removed empty/repeated texts:", original_count - len(sms))
print("Clean messages:", len(sms))

X_train, X_temp, y_train, y_temp = train_test_split(
    sms["text"], sms["label"], test_size=0.30,
    stratify=sms["label"], random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50,
    stratify=y_temp, random_state=42
)
print("Train:", len(X_train), "Validation:", len(X_val), "Test:", len(X_test))
print("Spam share in training:", round((y_train == "spam").mean() * 100, 1), "%")
print("Spam share in test:", round((y_test == "spam").mean() * 100, 1), "%")

## Step 4 — Build and train the classifier
**WHY?** Raw text must become numerical features before a standard classifier can learn from it.

- **TF-IDF:** Converts text into word-based numerical features.
- **Multinomial Naive Bayes:** Learns patterns associated with `ham` and `spam`.
- **Pipeline:** Keeps both steps together for training and future predictions.

**Ask:** Why train the entire pipeline on **training data only**?

**RUN:** Build and fit the pipeline. No need to understand Naive Bayes mathematics.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

model = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("classifier", MultinomialNB())
])
model.fit(X_train, y_train)
print("Training complete; classes:", model.classes_.tolist())

## Step 5 — Evaluate on the validation set
**WHY?** We must examine performance on messages the model did not train on.

**Ask:** Does high overall accuracy mean we detected most spam?

**RUN:** Compare the model with an intentionally bad **always-ham** baseline. Interpret spam precision and spam recall.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score

def show_scores(title, actual, predicted):
    print(title)
    print("  Accuracy:", round(accuracy_score(actual, predicted), 3))
    print("  Spam precision:", round(precision_score(
        actual, predicted, pos_label="spam", zero_division=0), 3))
    print("  Spam recall:", round(recall_score(
        actual, predicted, pos_label="spam", zero_division=0), 3))

val_predictions = model.predict(X_val)
show_scores("Our model — validation", y_val, val_predictions)
show_scores("Always-ham — validation", y_val, ["ham"] * len(y_val))

### Interpreting the validation results

| Metric | Our model | Always-ham baseline |
|---|---:|---:|
| Accuracy | 0.948 | 0.875 |
| Spam precision | 1.000 | 0.000 |
| Spam recall | 0.588 | 0.000 |

**What this means:** Our model is correct for 94.8% of validation messages. Its spam precision of 1.0 means every message it flagged as spam was truly spam in this validation set. Its recall of 0.588 means it found 58.8% of the real spam messages, so it still missed some spam. The always-ham baseline has high accuracy only because most messages are ham; it detects no spam.


### Guided Exercise 3 — Investigate a mistake
**WHY?** Metrics tell us *how much* the model errs; examples help us understand *how*.

**RUN:** Display errors. Change `kind` from `"missed spam"` to `"false alarm"` to inspect the other mistake type.

**Ask:** Which mistake can incorrectly hide a legitimate appointment reminder? Which mistake lets spam through?

**Note:** The public dataset contains historical real SMS. Discuss errors without copying messages into other systems.

In [ ]:
validation_review = pd.DataFrame({
    "text": X_val.to_numpy(),
    "actual": y_val.to_numpy(),
    "predicted": val_predictions
})
validation_review = validation_review[validation_review["actual"] != validation_review["predicted"]]
print("Validation mistakes:", len(validation_review))
print(validation_review.groupby(["actual", "predicted"]).size())

kind = "missed spam"  # YOUR TURN: change to "false alarm"
if kind == "missed spam":
    reviewed = validation_review.query("actual == 'spam' and predicted == 'ham'")
else:
    reviewed = validation_review.query("actual == 'ham' and predicted == 'spam'")
display(reviewed.head(3))

---
## OPTIONAL ADVANCED — Can we improve spam precision?

**Do this before the final test.** We will experiment **only on validation data**.

**Leading question:** What happens if we flag spam only when the model's spam score is higher?

We compare `0.50` with `0.80`. We are **not retraining**; we are changing the decision rule. Higher thresholds can reduce false alarms but may also miss more spam. Precision is **not guaranteed** to increase.

**Important:** A Naive Bayes `predict_proba` result is a model probability estimate, **not automatically a calibrated confidence**.

In [ ]:
spam_index = list(model.classes_).index("spam")
spam_scores = model.predict_proba(X_val)[:, spam_index]

def with_threshold(scores, threshold):
    return ["spam" if score >= threshold else "ham" for score in scores]

threshold_results = []
for threshold in [0.50, 0.80]:
    result = with_threshold(spam_scores, threshold)
    threshold_results.append({
        "Threshold": threshold,
        "Spam alerts": result.count("spam"),
        "Accuracy": round(accuracy_score(y_val, result), 3),
        "Spam precision": round(precision_score(y_val, result, pos_label="spam", zero_division=0), 3),
        "Spam recall": round(recall_score(y_val, result, pos_label="spam", zero_division=0), 3),
    })

display(pd.DataFrame(threshold_results))


### Interpreting the threshold comparison

| Threshold | When the model predicts `spam` | Typical effect |
|---:|---|---|
| `0.50` | Spam score is 0.50 or higher | More messages can be flagged as spam. |
| `0.80` | Spam score is 0.80 or higher | Fewer messages are flagged; some spam may be missed. |

**What this means:** The threshold is the rule used *after* training to turn a spam score into a label. Raising it from `0.50` to `0.80` makes the classifier stricter: a message needs a higher spam score before it is flagged. This often reduces false alarms and can increase precision, but it can also reduce recall because real spam is missed. Use the table above to compare the actual validation results; do not choose a threshold using the final test set. A Naive Bayes score is not automatically a calibrated probability or confidence.


### Advanced exercise — Experiment with strictness
**Your turn:** Change `strict_threshold` from `0.80` to `0.90` and rerun.

**Discuss:** Did precision increase? Did recall fall? Did the number of alerts change? Why might fewer alerts be better when messages are automatically blocked?

**Do not use test results to choose a threshold.** We will save the original baseline pipeline for the next session.

In [ ]:
strict_threshold = 0.80  # YOUR TURN: try 0.90
strict_predictions = with_threshold(spam_scores, strict_threshold)
show_scores("Strict validation rule", y_val, strict_predictions)
print("Spam alerts:", strict_predictions.count("spam"))

---
## Step 6 — Final evaluation on untouched test messages
**WHY?** The test set gives the selected baseline model a separate final check.

**RUN:** Evaluate the **original** trained pipeline, not a threshold chosen by looking at test results.

**Read the confusion matrix:** Rows are actual labels and columns are predicted labels. For spam detection, ham → spam is a **false positive**; spam → ham is a **false negative**.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt

test_predictions = model.predict(X_test)
show_scores("Final test — original pipeline", y_test, test_predictions)
ConfusionMatrixDisplay.from_predictions(
    y_test, test_predictions, labels=["ham", "spam"],
    display_labels=["Ham", "Spam"], values_format="d"
)
plt.title("Final Test — Actual (rows) vs Predicted (columns)")
plt.show()

## Step 7 — Predict your own SMS messages
**WHY?** This is inference: reuse our trained pipeline on fresh text.

### Guided Exercise 4 — Create 3 synthetic messages
- A clearly legitimate SMS.
- A clearly suspicious spam SMS.
- An ambiguous SMS (could reasonably be either).

**Your turn:** Replace the three example messages below with your own **invented** messages. Compare your expectation with the output.

**Remember:** Three messages are a classroom demonstration, **not** a reliable test dataset or accuracy measurement.

In [ ]:
new_messages = [
    "Are we meeting after class tomorrow?",      # YOUR TURN: write a ham example
    "WIN a FREE PRIZE! Claim your reward now!", # YOUR TURN: write a spam example
    "Your parcel is ready to collect today."   # YOUR TURN: write an ambiguous example
]

for message, predicted in zip(new_messages, model.predict(new_messages)):
    print(f"{predicted.upper():4} | {message}")

## Step 8 — Save the **entire** trained pipeline
**WHY?** A saved classifier alone would not know how to transform SMS text into the same features. Save TF-IDF **and** Naive Bayes together.

**RUN:** Save a model artifact and a small JSON record of its origin and software versions. Verify that a freshly loaded *trusted local artifact* produces the same predictions.

**For Session 3:** Keep the `.joblib` and `.json` files.

In [ ]:
import joblib
import json
from pathlib import Path

model_path = "sms_spam_pipeline.joblib"
info_path = "sms_spam_model_info.json"
joblib.dump(model, model_path)
info = {
    "dataset": "UCI SMS Spam Collection",
    "dataset_url": "https://archive.ics.uci.edu/dataset/228/sms+spam+collection",
    "source_records": original_count,
    "clean_records": len(sms),
    "pipeline": "TfidfVectorizer + MultinomialNB",
    "split_seed": 42,
    "saved_model": "baseline (no threshold customization)",
    "pandas_version": pd.__version__,
    "sklearn_version": sklearn.__version__,
    "joblib_version": joblib.__version__
}
Path(info_path).write_text(json.dumps(info, indent=2), encoding="utf-8")

trusted_local_copy = joblib.load(model_path)  # Load ONLY files you created/trust
assert list(trusted_local_copy.predict(new_messages)) == list(model.predict(new_messages))
print("Saved and verified:", model_path, "+", info_path)

# In Google Colab, uncomment these to download both files:
# from google.colab import files
# files.download(model_path)
# files.download(info_path)

## Wrap-up — Can you explain the whole workflow?
1. Why do we inspect the dataset before training?
2. Why must we split data before fitting TF-IDF?
3. Why can high accuracy coexist with poor spam detection?
4. What is the difference between a false positive and false negative?
5. Why save the whole pipeline rather than only Naive Bayes?

**Next session:** MLflow — comparing and tracking experiments and saved model versions.

**Useful sources:** [UCI SMS collection](https://archive.ics.uci.edu/dataset/228/sms+spam+collection) · [scikit-learn text features](https://scikit-learn.org/stable/modules/feature_extraction.html#text-feature-extraction) · [Pipeline API](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html) · [Model persistence](https://scikit-learn.org/stable/model_persistence.html)